In [ ]:
import cv2
import pypdfium2 as pdfium
import matplotlib.pyplot as plt
import numpy as np
import os

import scipy.signal

FILE = "example_filled.pdf"
SCALE = 5.0

doc = pdfium.PdfDocument(FILE)
page = doc[0]
print("Page size:", doc.get_page_size(0))

width = int(doc.get_page_size(0)[0] * SCALE)
height = int(doc.get_page_size(0)[1] * SCALE)

print("Width:", width)
print("Height:", height)

image = page.render(scale = SCALE,no_smoothimage=True,optimize_mode="print")
image = image.to_numpy()

#read the FILE as a tiff image
#image = cv2.imread(FILE)
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
#show the image using matplotlib in its original size

plt.imshow(image)
plt.show()

In [ ]:
#show only part of the image trimming by width
plt.imshow(image[:,int(width*0.72):int(width*0.87)])
plt.show()
plt.imshow(image[:,int(width*0.525):int(width*0.675)])
plt.show()
plt.imshow(image[:,int(width*0.33):int(width*0.478)])
plt.show()
plt.imshow(image[:,int(width*0.14):int(width*0.29)])
plt.show()

In [3]:
def straighten_image(original_image,**kwargs):
    threshold = kwargs.get("threshold", 40)
    image_percent = kwargs.get("image_percent", 0.05)
    image = cv2.cvtColor(original_image, cv2.COLOR_BGR2GRAY)
    height = image.shape[0]

    _, thresh = cv2.threshold(image, threshold, 255, cv2.THRESH_BINARY)
    thresh = cv2.bitwise_not(thresh)
    linesTop = cv2.HoughLinesP(thresh[0:int(height*image_percent)],1, np.pi/180, 100, minLineLength=5, maxLineGap=100) #N.B., 5% here
    linesBottom = cv2.HoughLinesP(thresh[int(height-height*image_percent):],1, np.pi/180, 100, minLineLength=20, maxLineGap=100)
    
    lines = np.concatenate((linesTop, linesBottom))
    angles = []
    for line in lines:
        x1, y1, x2, y2 = line[0]
        angle = np.arctan2(y2 - y1, x2 - x1) * 180 / np.pi
        angles.append(angle)

    angle = np.mean(angles)
    
    image = cv2.warpAffine(original_image, cv2.getRotationMatrix2D((image.shape[1]//2, image.shape[0]//2), angle, 1), (image.shape[1], image.shape[0]))
    return image
    

def find_black_bars(orig_image, **kwargs):
    
    threshold = kwargs.get("threshold", 127)
    right_scan_percent = kwargs.get("right_scan_percent", 0.03) 
    num_black_Bars = kwargs.get("num_black_Bars", 44)
    width = orig_image.shape[1]

    image = cv2.cvtColor(orig_image, cv2.COLOR_BGR2GRAY)

    _, thresh = cv2.threshold(image, threshold, 255, cv2.THRESH_BINARY)

    blackBars = []
    foundTop = False
    for i in range(0,thresh.shape[0]):
        if thresh[i,int(-width*right_scan_percent)] == 0 and not foundTop: #N.B., 3% here
            foundTop = True
            top = i
        if thresh[i,int(-width*right_scan_percent)] == 255 and foundTop:
            foundTop = False
            blackBars.append((top, i))
    
    if len(blackBars) != num_black_Bars: 
        return None
    return blackBars

def prepare_image(image, **kwargs):
    new_image = straighten_image(image, **kwargs)
    blackBars = find_black_bars(new_image, **kwargs)
    if blackBars == None:
        new_image = cv2.rotate(new_image, cv2.ROTATE_180)
        blackBars = find_black_bars(new_image, **kwargs)
    if blackBars == None:
        return None,None
    return new_image, blackBars    

In [4]:
image_prep,bars = prepare_image(image)



In [ ]:
def ssd(a,b):
    return np.sum((a-b)**2)

def load_number_symbols(**kwargs):
    path = kwargs.get("path", "")
    symbol_name = kwargs.get("symbol_name", "average_symbol.png")
    symbols = {}
    for i in range(10):
        symbols[i] = cv2.imread(os.path.join(path, f"{i}",symbol_name), cv2.IMREAD_GRAYSCALE)
    return symbols

def scan_for_symbol(orig_line, symbol, **kwargs):
    '''returns the coordinates of the symbol in the line. The symbol can appear multiple times'''

    #prepare the line and the symbol
    threshold = kwargs.get("threshold", 200)
    erode_iterations = kwargs.get("erode_iterations", 1)

    if len(orig_line.shape) == 3:
        orig_line = cv2.cvtColor(orig_line, cv2.COLOR_BGR2GRAY)
    
    _, line = cv2.threshold(orig_line, threshold, 255, cv2.THRESH_BINARY)

    kernel = np.ones((3,3),np.uint8)
    line = cv2.erode(line,kernel,iterations=erode_iterations)
    
    #check if symbol is black and white, if color, convert to grayscale
    if len(symbol.shape) == 3:
        symbol = cv2.cvtColor(symbol, cv2.COLOR_BGR2GRAY)
    _,symbol = cv2.threshold(symbol, threshold, 255, cv2.THRESH_BINARY)
    symbol = cv2.erode(symbol,kernel,iterations=erode_iterations)

    #scan the line for the symbol
    ssd_values = np.zeros([line.shape[1]])
    ssd_values.fill(1000)
    for x in range(symbol.shape[1]//2,1+((line.shape[1]-symbol.shape[1]//2))):
        patch = line[:,x-symbol.shape[1]//2:x+symbol.shape[1]//2]
        ssd_values[x] = ssd(patch,symbol)
    
    #find centres of the symbols
    centres = []
    for pos_thresh in range(500,1,-1):
        centres = []
        #set new_ssd_values to 0 if the ssd value is below the threshold, and the value otherwise
        new_ssd_values = np.where(ssd_values < pos_thresh, 0, 1)
        found_start = False
        start_pos = 0
        for x in range(len(new_ssd_values)):
            if new_ssd_values[x] == 0 and not found_start:
                found_start = True
                start_pos = x
            if new_ssd_values[x] != 0 and found_start:
                found_start = False
                centres.append((start_pos+x)//2)
        if len(centres) == 8:
            break
    return centres
       

def get_matriculation_number(image,blackBars,symbols,**kwargs):
    area = kwargs.get("area_start",0.75), kwargs.get("area_end",0.96)
    thresh_background = kwargs.get("thresh_background",200)
    variance_threshold = kwargs.get("variance_threshold",0.1)
    
    width = image.shape[1]
    matriculation_number = []

    center_bin = np.zeros([8])
    center_count = 0
    
    for l in range(2,12): #we consider the 2nd-11th black bar
        line = image[blackBars[l][0]:blackBars[l][1],
                     int(width*area[0]):int(width*area[1])].copy()
        
        #each line contains a faint background color in some areas, we need to turn that white.
        line = cv2.cvtColor(line, cv2.COLOR_BGR2GRAY)
        _, line = cv2.threshold(line, thresh_background, 255, cv2.THRESH_BINARY)
        
        #if the line isn't as high as the symbol, pad the line with white pixels at the top
        if line.shape[0] < symbols[l-2].shape[0]:
            line = cv2.copyMakeBorder(line, symbols[0].shape[0]-line.shape[0],0,0,0,cv2.BORDER_CONSTANT,value=255)
        
        #if the symbol is narrower than the line, pad the symbol with white pixels at the top
        if symbols[l-2].shape[0] < line.shape[0]:
            symbols[l-2] = cv2.copyMakeBorder(symbols[l-2], line.shape[0]-symbols[l-2].shape[0],0,0,0,cv2.BORDER_CONSTANT,value=255)

        centres = scan_for_symbol(line, symbols[l-2])
        if len(centres) == 8:
            center_count+=1
            for c in range(8):
                center_bin[c] += centres[c]
        #print(l-2,centres)

    center_bin/=center_count
    #round the centers to the nearest integer
    center_bin = np.round(center_bin).astype(int)

    #we now have the center coordinates for each column.
    #we are going to scan down each column to find the number
    for x in range(8):
        lowest_color = 1000000
        lowest_value = None
        average_value = 0
        for y in range(10):
            #extract the patch from the image
            line = image[blackBars[y+2][0]:blackBars[y+2][1],int(width*area[0]):int(width*area[1])].copy()
            #set the red channel to 255
            #line[:,:,1] = 255
            patch = line[:,center_bin[x]-symbols[0].shape[1]//2:center_bin[x]+symbols[0].shape[1]//2]
            #patch = cv2.cvtColor(patch, cv2.COLOR_BGR2GRAY)
            _, patch = cv2.threshold(patch, thresh_background, 255, cv2.THRESH_BINARY)
            #convert patch to HSV
            patch = cv2.cvtColor(patch, cv2.COLOR_BGR2HSV)
            
            #the color of the patch is the sum of the pixels
            patch_color = np.sum(patch)
            average_value += patch_color
            if patch_color < lowest_color:
                lowest_color = patch_color
                lowest_value = y
        matriculation_number.append(lowest_value)
        if (average_value/10 - lowest_color)/(average_value/10) < variance_threshold:
            return None
    return matriculation_number
    

get_matriculation_number(image_prep,bars,load_number_symbols())



In [6]:
def get_mark_indexes(line,area=None,**kwargs):
    answers = []
    num_dilations = kwargs.get("num_dilations", 2)
    h_thresh = kwargs.get("h_thresh", 90)
    s_thresh = kwargs.get("s_thresh", 90)
    v_thresh = kwargs.get("v_thresh", 200)    
    window_width = kwargs.get("window_width", 60) #the width of the letters on the answer sheet
    #line is a color image

    line = line[:,int(line.shape[1]*area[0]):int(line.shape[1]*area[1])]
    line = cv2.cvtColor(line, cv2.COLOR_BGR2HSV)
    h = line[:,:,0]
    s = line[:,:,1]
    v = line[:,:,2]
    _,h = cv2.threshold(h, h_thresh, 255, cv2.THRESH_BINARY)
    _,s = cv2.threshold(s, s_thresh, 255, cv2.THRESH_BINARY)
    _,v = cv2.threshold(v, v_thresh, 255, cv2.THRESH_BINARY)

    mark_mask = np.logical_and(h == 255, v == 0)
    mark_mask_location = np.where(mark_mask, 255, 0)
    
    mark_mask_location = mark_mask_location.astype(np.uint8)
    #dilate the mask location
    
    mark_mask_location = cv2.dilate(mark_mask_location, np.ones((3,3),np.uint8), iterations=num_dilations)

    #we now introduce a new mask on the h and s channels
    hsmask = np.logical_and(h == 255, s == 255)
    hsmask_location = np.where(hsmask, 255, 0)
    hsmask_location = hsmask_location.astype(np.uint8) 
    #dilate hsmask_location
    hsmask_location = cv2.dilate(hsmask_location, np.ones((3,3),np.uint8), iterations=num_dilations)

    #the mark_mask_location will have a strong white area where the student has marked the answer
    #the hsmask_location will have a strong white area where the answer letters are.

    #try add blur to both
    mark_mask_location = cv2.GaussianBlur(mark_mask_location,(7,7),0)
    hsmask_location = cv2.GaussianBlur(hsmask_location,(7,7),0)
    
    mask_value = np.zeros([line.shape[1]-window_width]) #this is the mask for the hsmask_location containing the multiple choice letters
    ans_mask_value = np.zeros([line.shape[1]-window_width]) #this is the mask for the mark_mask_location containing the student answers

    #count number of white pixels in the window for the two masks and fill them in.
    for x in range(line.shape[1]-window_width//2,window_width//2,-1):
        window = hsmask_location[:,x-window_width//2:x+window_width//2]
        mask_value[x-1-window_width//2] = np.sum(window)

        window = mark_mask_location[:,x-window_width//2:x+window_width//2]
        ans_mask_value[x-1-window_width//2] = np.sum(window)
    
    answer_peaks = scipy.signal.find_peaks(ans_mask_value,distance=50,height=200000)
    two_peaks = scipy.signal.find_peaks(mask_value,distance=50,height=100000)
    
    for i in range(len(two_peaks[0])):    
        closest_distance = 100000
        for j in range(len(answer_peaks[0])):
            distance = abs(two_peaks[0][i]-answer_peaks[0][j])
            if distance < closest_distance:
                closest_distance = distance
    
        if closest_distance < 10:
            answers.append(i)
        else:
            continue
        #add the answers to the answer list as a tuple containing the index of True answers 
    return answers
    

In [ ]:
def get_matriculation_number(image,bars,**kwargs):
    matriculation_number = 0
    area = kwargs.get("matriculation_number_area", (0.75,0.96))
    for i in range(2,12):
        line = image[bars[i][0]:bars[i][1],:].copy()
        answers = get_mark_indexes(line,area=area,**kwargs)
        #print(answers)
        for a in answers:
            #print("setting digit ",7-a," to ",i-2)
            matriculation_number += (i-2)*10**(7-a)
    #turn matriculation number into a 7 digit string with 0 padding as needed
    return str(matriculation_number).zfill(8)


get_matriculation_number(image_prep,bars)
get_all_answers(image_prep,bars)


In [ ]:
def get_answers(line,**kwargs):
    answer_bounds = kwargs.get("answer_bounds", [[0.14,0.29],[0.33,0.478],[0.525,0.675],[0.72,0.87]])

    answers = []
    for a in answer_bounds:
        answers.append(get_mark_indexes(line,area=a,**kwargs))

    return answers

def get_all_answers(image,bars,**kwargs):
    answer_map = {}
    
    for i in range(12,42):
        line = image[bars[i][0]:bars[i][1],:].copy()
        answers = get_answers(line, **kwargs)
        #answers are for questions (i-11),(i-11)+30,(i-11)+60 and (i-11)+90
        answer_map[i-11] = answers[0]
        answer_map[i-11+30] = answers[1]
        answer_map[i-11+60] = answers[2]
        answer_map[i-11+90] = answers[3]
        #print("answers for ",i-11,i-11+30,i-11+60,i-11+90,":",answers)
    return answer_map
        
#line = image[bars[32][0]:bars[32][1],int(image.shape[1]*0.05):int(image.shape[1]*(1-0.05))].copy()
#line = image[bars[32][0]:bars[32][1],:].copy()
#print(get_answers(line))
#a = get_all_answers(image_prep,bars)
#print(a)


In [ ]:
import random

orig_image = image.copy()

if random.random() < 0.5:
    orig_image = cv2.rotate(orig_image, cv2.ROTATE_180)

plt.imshow(orig_image)
plt.show()

red_image = orig_image.copy()

#set anything where green is more than 100 or blue is more than 100 to 0 for that channel
red_image[(red_image[:,:,1] > 120) | (red_image[:,:,2] > 120)] = 0
red_image = cv2.cvtColor(red_image, cv2.COLOR_RGB2GRAY)
red_image = cv2.threshold(red_image, 100, 255, cv2.THRESH_BINARY)[1]

lines = cv2.HoughLinesP(red_image, 1, np.pi / 180, 1000, minLineLength=900, maxLineGap=0)




inverted = False
for line in lines:
    if line[0][1] < image.shape[1] / 2:
        inverted = True

    red_image = orig_image.copy()

#draw the lines
for line in lines:
    x1, y1, x2, y2 = line[0]
    cv2.line(red_image, (x1, y1), (x2, y2), (0,255,0),5)

#show the image

if inverted:
    red_image = cv2.rotate(red_image, cv2.ROTATE_180)


plt.imshow(red_image)

print(len(lines))


In [ ]:
#convert image to HSV
_, thresh = cv2.threshold(image, 200, 255, cv2.THRESH_BINARY)
hsv_image = cv2.cvtColor(thresh, cv2.COLOR_RGB2HSV)
plt.imshow(hsv_image)
plt.show()
plt.imshow(hsv_image[:,:,1], cmap="gray")
plt.show()

In [ ]:
#make the image greyscale
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
plt.imshow(gray, cmap='gray')

In [ ]:
#threshold the image, setting anything above 40 to 255
_, thresh = cv2.threshold(gray, 160, 255, cv2.THRESH_BINARY)
thresh = cv2.bitwise_not(thresh)
plt.imshow(thresh, cmap='gray')

In [4]:
#detect horizontal line which should run above pixel 50 and below pixel 800 in the thresh image and draw it on the original image. Use the ransac algorithm to detect the line

#invert the image

linesTop = cv2.HoughLinesP(thresh[0:int(height*0.05)],1, np.pi/180, 100, minLineLength=5, maxLineGap=100) #N.B., 5% here
linesBottom = cv2.HoughLinesP(thresh[int(height-height*0.05):],1, np.pi/180, 100, minLineLength=5, maxLineGap=100)

lines = np.concatenate((linesTop, linesBottom))

#print(len(lines))
#for line in lines:
#    x1, y1, x2, y2 = line[0]
#    cv2.line(image, (x1, y1), (x2, y2), (0, 0, 0), 5)

#plt.imshow(image)



In [ ]:
#get the average angle of all the lines and rotate the image to straighten the image
angles = []
for line in lines:
    x1, y1, x2, y2 = line[0]
    angle = np.arctan2(y2 - y1, x2 - x1) * 180 / np.pi
    angles.append(angle)

angle = np.mean(angles)
print(angle)

rotated = cv2.warpAffine(image, cv2.getRotationMatrix2D((image.shape[1]//2, image.shape[0]//2), angle, 1), (image.shape[1], image.shape[0]))

plt.imshow(rotated)

In [73]:
#we need to make the black bars properly black via thresholding
gray = cv2.cvtColor(rotated, cv2.COLOR_BGR2GRAY)
_, thresh = cv2.threshold(gray, 160, 255, cv2.THRESH_BINARY)

#move down the 15h pixel along the right hand side and detect black bars, recording their start and finish points

foundTop = False

blackBars = []
for i in range(0,thresh.shape[0]):
    if thresh[i,int(-width*0.03)] == 0 and not foundTop: #N.B., 3% here
        foundTop = True
        top = i
    if thresh[i,int(-width*0.03)] == 255 and foundTop:
        foundTop = False
        blackBars.append((top, i))

#draw horizontal lines across the image along the centre of each black bar
#for b in blackBars:
#    middle = (b[0] + b[1]) // 2
#    line = np.array([[0, middle], [thresh.shape[1], middle]])
#    cv2.line(rotated, tuple(line[0]), tuple(line[1]), (0, 0, 255), 2)

#plt.imshow(rotated)
        

        

In [7]:
def ssd(a,b):
    return np.sum((a-b)**2)

In [ ]:
#let's try erode the lines and see what happens
k = 3
line = rotated[blackBars[k][0]:blackBars[k][1], int(width*0.75):int(width*0.96)].copy()

line = cv2.cvtColor(line, cv2.COLOR_BGR2GRAY)
_, line = cv2.threshold(line, 200, 255, cv2.THRESH_BINARY)
for i in range(7):
    line = cv2.erode(line, np.ones((3,3)))
    plt.imshow(line)
    plt.show()

In [ ]:
#extract "template" symbol which we will use to find other symbols.
for i in range(2,12):
    line = rotated[blackBars[i][0]:blackBars[i][1], int(width*0.75):int(width*0.95)]
    line = cv2.cvtColor(line, cv2.COLOR_BGR2GRAY)
    line = cv2.threshold(line, 200, 255, cv2.THRESH_BINARY)[1]
    bwline = line.copy() #used later for writing the symbol
    
    #uncomment to generate symbols
    symbol=rotated[blackBars[i][0]:blackBars[i][1], int(width*0.75)+26:int(width*0.75)+86]
    symbol = cv2.cvtColor(symbol, cv2.COLOR_BGR2GRAY)
    symbol = cv2.threshold(symbol, 200, 255, cv2.THRESH_BINARY)[1]
    
    #uncomment to use averaged symbols from pregenerated data
    #symbol = cv2.imread(f"{str(i-2)}/average_symbol.png",0)
    #symbol = cv2.threshold(symbol, 220, 255, cv2.THRESH_BINARY)[1]

    kernel = np.ones((3,3),np.uint8)
    
    symbol = cv2.erode(symbol,kernel,iterations = 2)
    line = cv2.erode(line,kernel,iterations = 2)

    #plt.imshow(line, cmap='gray')
    #plt.show()
    #plt.imshow(symbol, cmap='gray')
    #plt.show()

    ssd_values = np.zeros([line.shape[1]])
    ssd_values.fill(1000)
    for x in range(symbol.shape[1]//2,1+((line.shape[1]-symbol.shape[1]//2))):
        patch = line[:,x-symbol.shape[1]//2:x+symbol.shape[1]//2]
        ssd_values[x] = ssd(patch,symbol)

    #print(ssd_values)
    
    #ssd_values now contains the ssd values for each position of the symbol in the line.

    #find optimal ssd values which will give us 8 non-zero values
    centres = []
    for pos_thresh in range(500,1,-1):
        centres = []
        #set new_ssd_values to 0 if the ssd value is below the threshold, and the value otherwise
        new_ssd_values = np.where(ssd_values < pos_thresh, 0, 1)
        found_start = False
        start_pos = 0
        for x in range(len(new_ssd_values)):
            if new_ssd_values[x] == 0 and not found_start:
                found_start = True
                start_pos = x
            if new_ssd_values[x] != 0 and found_start:
                found_start = False
                centres.append((start_pos+x)//2)
        if len(centres) == 8:
            break
    print(centres)
    
    #so now we have the centres of each symbol. We want to write the symbol at each centre on the original line to the file. We want to put each one in a directory "i-2" as we are starting from 2.
    if not os.path.exists(str(i-2)):
        os.mkdir(str(i-2))
    average_symbol = np.zeros([symbol.shape[0],symbol.shape[1]])
    for x in centres:
        symbol2 = bwline[:,x-symbol.shape[1]//2:x+symbol.shape[1]//2]
        average_symbol += symbol2
        cv2.imwrite(f"{str(i-2)}/symbol"+str(i)+str(x)+".png",symbol2)
        cv2.line(rotated, (int(width*0.75)+x-symbol2.shape[1]//2, blackBars[i][0]), (int(width*0.75)+x-symbol2.shape[1]//2, blackBars[i][1]), (0, 0, 255), 2)
    average_symbol = average_symbol / 8
    cv2.imwrite(f"{str(i-2)}/average_symbol.png",average_symbol)

        
                       


In [ ]:
from tqdm import tqdm

symbols = []
for i in range(10):
    symbol = cv2.imread(f"{str(i)}/average_symbol.png")
    symbol = cv2.cvtColor(symbol, cv2.COLOR_BGR2GRAY)
    symbols.append(symbol)

out = cv2.cvtColor(rotated, cv2.COLOR_BGR2GRAY)
#out = cv2.threshold(out, 160, 255, cv2.THRESH_BINARY)[1]

#for each of the lines we want to find the symbols in the line and display them
for i in range(2,12):
    line = rotated[blackBars[i][0]:blackBars[i][1], int(width*0.75):int(width*0.95)]
    line = cv2.cvtColor(line, cv2.COLOR_BGR2GRAY)
    line = cv2.threshold(line, 200, 255, cv2.THRESH_BINARY)[1]

    





In [ ]:
best_j=0 #this is the threshold for ssd
for i in range(2,12):
    line = rotated[blackBars[i][0]:blackBars[i][1], int(width*0.75):int(width*0.95)]
    line = cv2.threshold(line, 200, 255, cv2.THRESH_BINARY)[1]
    print(line.shape)
    symbol=rotated[blackBars[i][0]:blackBars[i][1], int(width*0.75)+26:int(width*0.75)+86]
    symbol = cv2.threshold(symbol, 200, 255, cv2.THRESH_BINARY)[1]
    print(symbol.shape)

    #erode the symbol to make it more distinct
    kernel = np.ones((3,3), np.uint8)
    symbol = cv2.erode(symbol, kernel, iterations=1)

    #do the same for line
    line = cv2.erode(line, kernel, iterations=1)

    #now do a sum of squared differences between the line and the symbol
    #this will give us the best match
    
    ssd_values = []
    for x in range(symbol.shape[1]//2, 1+((line.shape[1]-symbol.shape[1]//2))):
        ssd = np.sum((line[:, (x-symbol.shape[1]//2):(x+symbol.shape[1]//2)] - symbol)**2)
        ssd_values.append(ssd)
        
    
    #normalise the ssd values to be between 0 and 1
    ssd_values = np.array(ssd_values)
    ssd_values = (ssd_values - np.min(ssd_values)) / (np.max(ssd_values) - np.min(ssd_values))
    ssd_values *=255
    
    
    #check all ssd_values between 1 and 255 such that we have a total of 8 sequences of 0s. We wan the lowest ssd_value for this.
    for j in range(1, 255):
        #look for 8 sequences of 0s
        num_found = 0
        found = False
        start = 0
        for x in range(0, len(ssd_values)):
            if ssd_values[x] <= j and not found:
                start = x
                found = True
            if ssd_values[x] > j and found:
                found = False
                num_found+=1

        if num_found == 8:
            break

    if best_j < j:
        best_j = j  

    ssd_values[ssd_values<best_j] = 0
    #print(ssd_values)

    #draw the ssd values on the line
    for x in range(symbol.shape[1]//2, ((line.shape[1]-symbol.shape[1]//2))):
        cv2.line(line, (x, 0), (x, int(ssd_values[x-symbol.shape[1]//2])), (0, 0, 255), 1)
    
    #make it greyscale
    line = cv2.cvtColor(line, cv2.COLOR_BGR2GRAY)
    symbol = cv2.cvtColor(symbol, cv2.COLOR_BGR2GRAY)
    plt.imshow(line)
    plt.show()
    plt.imshow(symbol, cmap='gray')
    plt.show()
    #print out the x coordinate of the minimum value of ssd_values, where a sequence of 0s returns only the central point
    found = False
    start = 0
    for x in range(0, len(ssd_values)):
        if ssd_values[x] == 0 and not found:
            start = x
            found = True
        if ssd_values[x] != 0 and found:
            found = False
            print(start + (x-start)//2)
    

print(best_j)

    

In [ ]:
best_j=0 #this is the threshold for ssd
for i in range(2,12):
    line = rotated[blackBars[i][0]:blackBars[i][1], int(width*0.75):int(width*0.95)]
    line = cv2.threshold(line, 200, 255, cv2.THRESH_BINARY)[1]
    print(line.shape)
    symbol=rotated[blackBars[i][0]:blackBars[i][1], int(width*0.75)+26:int(width*0.75)+86]
    symbol = cv2.threshold(symbol, 200, 255, cv2.THRESH_BINARY)[1]
    print(symbol.shape)

    #erode the symbol to make it more distinct
    kernel = np.ones((3,3), np.uint8)
    symbol = cv2.erode(symbol, kernel, iterations=1)

    #do the same for line
    line = cv2.erode(line, kernel, iterations=1)

    #now do a sum of squared differences between the line and the symbol
    #this will give us the best match
    
    ssd_values = []
    for x in range(symbol.shape[1]//2, 1+((line.shape[1]-symbol.shape[1]//2))):
        ssd = np.sum((line[:, (x-symbol.shape[1]//2):(x+symbol.shape[1]//2)] - symbol)**2)
        ssd_values.append(ssd)
        
    
    #normalise the ssd values to be between 0 and 1
    ssd_values = np.array(ssd_values)
    ssd_values = (ssd_values - np.min(ssd_values)) / (np.max(ssd_values) - np.min(ssd_values))
    ssd_values *=255
    
    
    #check all ssd_values between 1 and 255 such that we have a total of 8 sequences of 0s. We wan the lowest ssd_value for this.
    for j in range(1, 255):
        #look for 8 sequences of 0s
        num_found = 0
        found = False
        start = 0
        for x in range(0, len(ssd_values)):
            if ssd_values[x] <= j and not found:
                start = x
                found = True
            if ssd_values[x] > j and found:
                found = False
                num_found+=1

        if num_found == 8:
            break

    if best_j < j:
        best_j = j  

    ssd_values[ssd_values<best_j] = 0
    #print(ssd_values)

    for j in range(0, len(ssd_values)):
        if ssd_values[j]>0:
            print(i,j,ssd_values[j])
            #smb = line[:, (i-symbol.shape[1]//2):(i+symbol.shape[1]//2)]
            #plt.imshow(smb)
            #plt.show()


In [ ]:
bb